# Leaky modes of a step-index fiber

Step-index fibers typically have  guided modes and *leaky modes*. Leaky modes are also called *resonances* or quasinormal modes. They are defined as a nontrivial function 
$\varphi$  that satisfies, together with some complex number $\beta$, the following  Helmholtz equation on the entire plane containing a fiber cross section, 

$$
\Delta \varphi + k^2 n^2 \varphi = \beta^2 \varphi \qquad \text{ in all } \mathbb R^2.
$$

Instead of a boundary condition near the fiber boundary, leaky modes $\varphi$ satisfy the condition that it is **outgoing** at infinity.

The definition of an  outgoing solution of $\Delta \varphi + (k^2 n^2 -\beta^2) \varphi = 0$ when $\kappa^2 = k^2 n^2 -\beta^2$ is real is simple and can be found in textbooks. One equivalent characterization of outgoing $\varphi$ when $\kappa$ is real is that $\varphi$ satisfies the [Sommerfeld radiation condition](https://en.wikipedia.org/wiki/Sommerfeld_radiation_condition).  For complex $\kappa$ however, the definition of outgoing   is more delicate. A mathematically correct approach is to use complex analysis to define an outgoing $\varphi$ as a meromorphic continuation of an outgoing solution obtained in the real $\kappa$ case. Leaky modes have complex $\beta$. In our numerical methods to compute leaky modes,  the perfectly matched layer (PML)  selects outgoing solutions by damping them exponentially. In our semi-analytical methods, we select Hankel functions that are outgoing to match with fiber core solutions.

Connection between imaginary part of $\beta$ and confinement loss $\dots$

## Semi-analytical leaky mode finder

Leaky modes were computed in the non-dimensional $Z$-plane, while the guided modes were computed in the non-dimensional $X$-plane -- see explanation in notebook [docs/1.1](1_1_stepindex_exact.ipynb)  ... Is there an explanation there?


$X$ and $Z$ are related to each other through the fiber V-number:
$$
X^2 - Z^2 = V^2
$$

In [ ]:
from fibermode import StepIndexExact

In [ ]:
f = StepIndexExact('Nufern_Yb')

In [ ]:
z = f.leaky_propagation_constants(0)
np.array(z)

In [ ]:
X, Y, F, mode,  = f.visualize_leaky_mode(z[0], 0)

In [ ]:
import matplotlib.pyplot as plt
from matplotlib import cm
%matplotlib ipympl

In [ ]:
fig, ax = plt.subplots(1, 2, subplot_kw={'projection': '3d'})
fig.set_size_inches(10, 7)
fig.tight_layout()

ax[0].plot_surface(X, Y, F.real, cmap=cm.coolwarm, linewidth=0);
ax[0].view_init(azim=-60, elev=68)

ax[1].plot_surface(X, Y, F.imag, cmap=cm.coolwarm);
ax[1].view_init(azim=-60, elev=68)

## Numerical leaky mode finder

Using PML, the `StepIndex` class of `fibermode` provides numerical routines to compute the same leaky modes. 

Important:  the numerical modes are only accurate in the region without PML.

In [ ]:
from fibermode import StepIndex
from ngsolve.webgui import Draw

In [ ]:
fiber = StepIndex(fibername='Nufern_Yb', 
                  R=2, 
                  Rout=8,
                  refine=1)  # describe inputs in text please...

center = 1.96 - 0.19j  # center of circle to search for Z-resonance values
radius = 0.3  # search radius
p = 3  # polynomial degree

In [ ]:
fiber.Rout, fiber.R

In [ ]:
zh, y, yl, beta, _, _ = fiber.leakymode(p=3, rad=0.3, ctr=5.4-1.3j, alpha=5, verbose=False)

In [ ]:
yh = y.gridfun()
yh_real = -10 * yh.real

In [ ]:
Draw(yh_real, fiber.mesh, deformation=True, euler_angles=[-55, 10,10]);

Describe please (compare with the exact mode pattern before the onset of PML with the exact case $\dots$ etc.)

## Spectral locations

For an optical fiber, a typical diagram of propagation constant locations ($\beta$-values) in the complex plane 
takes the following form:

<div align="center"><img src='./figs/spectrumBeta.png' width="250" align="center"/></div>

In contrast, the non-dimensional $Z$ value locations in the complex plane have a different structure. Since $Z^2$ is a Schrödinger eigenvalue and since the essential spectrum of the Laplacian is unperturbed by a bounded potential well, the essential spectrum is marked in red. The guided modes are now on the imaginary axis in the $Z$-plane, while the leaky modes hover just below the real axis.

<div align="center"><img src='./figs/spectrumZ.png' width="250" align="center"/></div>

For the example fiber we've been considering, we now put together the computed $Z$ values and $\beta$ values. We will be able to discern the above spectral structure when we plot all the computed values together. 

In [ ]:
Z =[z]

In [ ]:
for i in range(1, 11):
  Z += [f.leaky_propagation_constants(i)]

In [ ]:
X = []
for i in range(3): 
  X += f.propagation_constants(i)

In [ ]:
Zguided2 = np.array(X)**2 - f.fiberV()**2
Zguided2

In [ ]:
Zguided = 1j  * np.sqrt(-Zguided2)
Zguided

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(8, 6)
ax.set_title('Spectrum of the optical fiber in the $Z$ plane')
ax.set_xlabel('real axis')
ax.set_ylabel('imaginary axis')

# draw real and imaginary axis of the complex plane
ax.plot([0, 0], [-2, 5], 'c');
ax.plot([-1, 13], [0, 0], 'c');

# plot the guided mode eigenvalues
ax.plot(Zguided.real, Zguided.imag, 'k*', label='Guided')

# plot the leakymode eigenvalues
for l in range(10):
    lab = 'Leaky(%d)' % l
    ax.scatter(np.array(Z[l]).real, np.array(Z[l]).imag, label=lab)
ax.grid(True); ax.legend();

The physical propagation constants are large and are located in different region. 

In [ ]:
betas = []
for z in Z:
    betas.append(f.ZtoBeta(np.array(z)))

In [ ]:
fig, ax = plt.subplots()
fig.set_size_inches(8, 6)
ax.set_title('Spectrum of the optical fiber in the $\\beta$ plane')
ax.set_xlabel('real part')
ax.set_ylabel('imaginary part')
ax.plot(f.XtoBeta(X), [0]*len(X), 'k*', label='Guided')
for l in range(10):
    lab = 'Leaky(%d)' % l
    ax.scatter(betas[l].real, betas[l].imag, label=lab)
ax.grid(True); ax.legend();